# Projet TSA

CANAL DE TRANSMISSION 5

### Imports

In [145]:
import numpy as np
import cv2

from scipy import signal
from scipy.fftpack import fft
from scipy.linalg import toeplitz

import pandas as pd
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import struct

import msicpe
print(msicpe.__version__)
from msicpe import tsa

1.1.0


*On rappelle que la documentation de la bibliothèque msicpe peut être trouvée en ligne --> __[msicpe](https://cpe.pages.in2p3.fr/msi/toolbox/msicpe.tsa.html)__.*

## Séance 1 - MDFB

### Chargement des données


In [146]:
Fs, s = msicpe.tsa.load_signal('audio')

# réduction de la dimension de s
s = s[:1500]

In [147]:
# propriétés du signal
N = 3    # nombre de points du signal s
D = len(s)            # durée du signal s

# vecteur temps
ts = np.linspace(0, D, N, endpoint=False)

In [148]:
# affichage du signal s
Npts_to_plot = N

df_s = pd.DataFrame({'x': ts[:Npts_to_plot], 'y': s[:Npts_to_plot], 'legende':'s'})
fig = px.line(df_s, x='x',  y='y', color='legende', labels={'x':'Temps (s)', 'y':''}, title='Signal temporel', width=700)
fig.show()

### Binarisation

In [149]:
# signal binarisé
dtype = 8 #1.25 int8/s
sb = msicpe.tsa.data2bin(s)
Nb = N*8            # nombre de points du signal sb, calculé à partir de N et du type des données

# vecteur temps correspondant
d_bin = 10       # frequence d'echantillonnage du signal binaire
tb = np.arange(Nb)/d_bin

## **ATTENTION**
L’affichage d’un signal possédant un très grand nombre de points peut faire crasher VSCode.
Aussi, pour chaque signal que vous souhaiterez afficher, pensez à **réduire sa dimension** **dans la dataframe créée pour l’affichage**.

In [150]:
# Affichage de s_b 
Nbits_to_plot = Nb

df_sb = pd.DataFrame({'x': tb[:Nbits_to_plot], 'y': sb[:Nbits_to_plot], 'legende':'sb'})
fig2 = px.scatter(df_sb, x='x',  y='y', color='legende', labels={'x': 'temps' , 'y':''}, title="Signal binarisé" , width=700 )
fig2.show()

### Modulation

In [151]:
# paramètres de modulation
A   = 5
nu0 = 20
nu1 = 40

# vecteur des instants d’échantillonnage du signal modulé s_m
F_mod = 1000
T_mod  = 1/F_mod
N_mod  = Nb * 100

tm = np.arange(N_mod) * T_mod

# signal modulé

sm = np.array([])
for k,val in enumerate(sb):
    if val == 0:
        sm = np.concatenate((sm, A*np.cos(2*np.pi*nu0*tm[k*100:(k+1)*100])))
    if val == 1:
        sm = np.concatenate((sm, A*np.cos(2*np.pi*nu1*tm[k*100:(k+1)*100])))


In [152]:
# Affichage de s_m
df_sm = pd.DataFrame({'x': tm[:N_mod], 'y': sm[:N_mod], 'legende': 'ee'})
fig3 = px.line(df_sm, x='x',  y='y', color='legende', labels={'x': 'Temps(s)' , 'y':'Amplitude(A)'}, title= 'Signal modulé' , width=700 )
fig3.show()

### Affichages

In [153]:
# Affichage de s_b et s_m sur une seule figure
fig4 = make_subplots(specs=[[{"secondary_y": True}]])

fig41 = px.line(df_sm, x='x',  y='y')
    
fig42 = px.scatter(df_sb, x='x', y='y')
fig42.update_traces(yaxis='y2', marker_color='#cc0000', marker_size=8)

fig4.add_traces(fig41.data + fig42.data)
fig4.update_layout(title='Signaux temporels', width=700,
                    xaxis =dict(title=dict(text='Temps (s)')), 
                    yaxis =dict(title=dict(text='signal modulé sm')), 
                    yaxis2=dict(title=dict(text='signal binaire sb',font=dict(color="#cc0000")), color='#cc0000'),
)
fig4.show()

## Séance 2 - Estimation de la Densité De Probabilité (DDP)

### Analyses _in-silico_

In [154]:
## Fonction de calcul d'histogramme
def histo(x,n=None,M=None):
    if n is not None :
        x = x[:n]
        
    n_pts = len(x)
    
    if M is None : 
        Delta_opt = 3.49 * np.std(x) * n_pts**(-1/3)
        M = int(np.ceil((x.max() - x.min()) / Delta_opt)) # l’arrondi supérieur np.ceil (mscicpe)
    
    Delta = (x.max() - x.min()) / M
    Ni, bords = np.histogram(x, bins=M, density=False)

    DDPest = Ni / (n_pts * Delta) # Question 2.4 de la préparation : Hi
    c = (bords[:-1] + bords[1:])/2 # Calcul des ci : donne le centre de chaque classe

    return DDPest, c, Delta


## Foncton d'affichage
def compareDDP(c, DDPest, cth, DDPth, std_estim, title=None):
    """Fonction d’affichage qui trace sur un même graphe les histogrammes et courbes suivants :
    - la DDP théorique du canal en fonction d'un vecteur de centroïdes cth, 
    - la DDP théorique plus ou moins l'écart type (estimé sur les données) du canal en fonction d'un vecteur de centroïdes cth, 
    - l'histogramme de la DDP estimée numériquement en fonction des centroïdes c,
    
    Args:
        c : vecteur des centroïdes utilisés pour le calcul de l'histogramme,
        DDPest : l'histogramme de la DDP estimée numériquement,
        cth : vecteur des centroïdes utilisés pour le calcul de la DDP théorique,
        DDPth : la DDP théorique du canal,
        std_estim : l'écart-type du canal (estimé à partir des données),
        title (opt) : titre de la figure
    """
    
    df_DDP_th                 = pd.DataFrame({'x':cth,  'y':DDPth,  'legende':'DDP_th',    'line_dash':'solid','color':'red'})
    df_DDP_th_plus_std_estim  = pd.DataFrame({'x':cth,  'y':DDPth + std_estim ,  'legende':'DDP_th+std','line_dash':'dash', 'color':'black'})
    df_DDP_th_minus_std_estim = pd.DataFrame({'x':cth,  'y':DDPth - std_estim,  'legende':'DDP_th-std','line_dash':'dash', 'color':'black'})
    df_DDP_th=pd.concat([df_DDP_th,df_DDP_th_plus_std_estim,df_DDP_th_minus_std_estim])
    
    fig = px.line(df_DDP_th, x='x',  y='y', title=title,
                  color='legende', color_discrete_sequence=['orangered','black','black'], 
                  line_dash='legende', line_dash_sequence=['solid', 'dash', 'dash'])
    fig.add_bar(x=c, y=DDPest, marker_color='gray', name="DDP estimée")
    fig.show()

In [155]:
# analyse du canal test
b = tsa.canal_test()

n = 1000
DDPest, c, Delta = histo(b, n=n, M=20)

cth = np.linspace(-5, 5, 1000)
DDPth = np.exp(-cth**2 / 2) / np.sqrt(2 * np.pi)
std_estim = np.sqrt(DDPth * (1 - Delta * DDPth) / (n * Delta))

compareDDP(c, DDPest, cth, DDPth, std_estim, title=f"Canal test : n = {n}, M = 20")

#### Influence de N

In [156]:
M=20
for n in [100, 1000, 10000, 200000]:
    DDPest, c, Delta = histo(b, n=n, M=M)
    std_estim = np.sqrt(DDPth * (1 - Delta * DDPth) / (n * Delta)) # Formule Question 2.6
    compareDDP(c, DDPest, cth, DDPth, std_estim,title=f"Influence de n = {n} : Delta = {Delta:.3f}")

#### Influence de $\Delta$

In [ ]:
N_fixed = 1000

for M in [1, 5, 20, None, 100, N_fixed]:
    DDPest, c, Delta = histo(b, n=N_fixed, M=M)
    std_estim = np.sqrt(DDPth*(1-Delta*DDPth)/(N_fixed*Delta))
    compareDDP(c, DDPest, cth, DDPth, std_estim, title='Validation sur le canal test avec N fixe et M variable')

### Analyses _in-situ_

In [158]:
# analyse du canal de transmission réel
canal_id = 
...      = tsa.transmit(... , canal_id)

...

SyntaxError: invalid syntax (2072990919.py, line 2)

## Séance 3 - Estimation de la Densité Spectrale de Puissance Moyenne

### Analyses _in-silico_

In [ ]:
def compareDSP(f,DSPth,DSPbiais,DSPest):
    """Fonction d’affichage commune aux trois estimateurs, qui trace sur un même graphe les DSP suivantes en dB 
    - la DSP théorique du canal, 
    - la DSP théoriquement obtenue avec un certain estimateur, 
    - et la DSP obtenue numériquement avec ce même estimateur.
    en fonction des fréquences réduites.
    
    Args:
        f : vecteur de fréquences réduites tel que 0 ≤ f < 0.5
        DSPth : la DSPM théorique vraie du canal test ΓX (f),
        DSPbiais : la DSPM théorique obtenue en utilisant l’estimateur ??? du canal test ΓX ∗W_{N,B}(f),
        DSPest : votre DSPM estimée dΓ_{???}(f),
    """
    
    pass

##### Estimateur spectral simple

In [ ]:
def estimateur_simple(x, nd, N, nfft):
    pass
    return DSPest1,f

##### Estimateur spectral moyenné

In [ ]:
def estimateur_moyenne(x, M, nfft):
    pass
    #signal.welch()
    return DSPest2,f

##### Estimateur spectral de Welch

In [ ]:
def estimateur_welch(x, window, M, Noverlap, nfft):
    pass
    #signal.welch()
    return DSPest3,f

### Analyses _in-situ_

In [ ]:
...

## Séance 4 - Détection d’un signal noyé dans un bruit

In [ ]:
# Transmission
sm_tr = tsa.transmit(sm, canal_id)

#### Débruitage par détection

In [ ]:
th =               # vecteur temps d'un symbole

# reponses impulsionnelles des filtres de détection
h0 = 
h1 = 

# filtrage en Fourier
Nfft = 

fm =        # vecteur de fréquences réduites
...

# signaux en sortie des 2 filtres
s0 = 
s1 =

# signal détecté
sd = ...
sd = sd.astype(int)


#### Étude de l’effet du bruit de transmission sur la précision

In [ ]:
def error(sd,sb):
    pass
    return e



#### Décodage

In [ ]:
s_tr = 

## Séance 5 - Prédiction AR d’ordre $\textit{M}$

#### Estimation de la fonction d’autocorrélation

In [ ]:
K = ...

Gam = signal.correlate(s_tr.astype(np.float32), s_tr.astype(np.float32), mode='full', method='auto')
Gam /= len(s_tr)
Gam = Gam[len(Gam)//2 - K:len(Gam)//2 + K + 1]
lags = signal.correlation_lags(len(s_tr), len(s_tr), mode="full")
lags = lags[len(lags)//2 - K:len(lags)//2 + K + 1]

#### Identification du modèle AR(M)

In [ ]:
M = 

# matrice du systeme lineaire 
G = toeplitz()

# second membre du systeme lineaire
b = 

# solution du systeme G.Phi = b
Phi = np.linalg.pinv(G) @ b

# coefficients du filtre de Wiener
h = 

# puissance de l'erreur
sigma = 

#### Prédiction Linéaire

In [ ]:
s_hat =

#### Restauration par filtrage causal/anti-causal

In [ ]:
...